In [1]:
import sys
!{sys.executable} -m pip install tensorflow

In [1]:
# Load the TensorBoard notebook extension
%load_ext tensorboard


In [14]:
from keras import optimizers,losses,regularizers,Input
import datetime
import tensorflow as tf
import numpy as np
from keras.models import Model
from keras.layers import SimpleRNN,LSTM,Dense
#MODEL.ADD(SimpleRNN(32))model.add(layers.GRU(32,
#dropout=0.2,
#recurrent_dropout=0.2,
#input_shape=(None, float_data.shape[-1])))
from get_data import get_from_file
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

In [40]:
(whole_class_dict,float_labels_dict,one_hot_decimals_dict)=get_from_file(flag=False)
train_data=float_labels_dict[:-1]
test_data=float_labels_dict[:-1]
train_data.sort()
test_data.sort()

In [41]:
X_train=np.array([item[1] for item in train_data])
Y_train=np.array([item[0] for item in train_data])
X_test=np.array([item[1] for item in test_data])
Y_test=np.array([item[0] for item in test_data])
print(len(Y_train))

196


In [42]:
def build_model(shape=(1,)):
    input_tensor=Input(shape=shape)
    inner_layer=Dense(32)(input_tensor)
    inner_layer=Dense(64)(inner_layer)
    inner_layer=Dense(64)(inner_layer)
    output_tensor=Dense(1)(inner_layer)
    model = Model(input_tensor, output_tensor)
    model.summary()
    return model
num_epochs=10

In [43]:
k_data=float_labels_dict
k_data.sort()
x_train=np.array([item[1] for item in k_data])
y_train=np.array([item[0] for item in k_data])
def k_fold_train(X_train,Y_train,k,shape):
    all_scores=[]
    all_mae_histories=[]
    num_validate=len(X_train)//k
    for fold in range(k):
        X_val=X_train[fold * num_validate: (fold + 1) * num_validate]
        y_val=X_train[fold * num_validate: (fold + 1) * num_validate]
        partial_x_train = np.concatenate([X_train[:fold * num_validate],
                                          X_train[(fold + 1) * num_validate:]],
                                          axis=0)
        partial_y_train = np.concatenate(
            [Y_train[:fold * num_validate],
             Y_train[(fold + 1) * num_validate:]],
             axis=0)
        model=build_model(shape)
        model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
        history=model.fit(partial_x_train, partial_y_train,
              epochs=12, batch_size=96, verbose=0,callbacks=[tensorboard_callback])
        val_mse, val_mae = model.evaluate(X_val, y_val, verbose=0)
        all_scores.append(val_mae)
        mae_history = history.history['mae']
        all_mae_histories.append(mae_history)
    print(f'all val mae scores mean -> {np.mean(all_scores)}')
    average_mae_history = [np.mean([x[i] for x in all_mae_histories]) for i in range(num_epochs)]
    print(f'average_mae_history->{average_mae_history}')
#k_fold_train(x_train,y_train,k=4,shape=(1,))

In [44]:
def train(k=4,xtrain=X_train,ytrain=Y_train,xtest=X_test,ytest=Y_test,shape=(1,)):
    #k_fold_train(k,X_train,Y_train,shape=(1,))
    model = build_model(shape)
    model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
    mean = xtrain.mean(axis=0)
    xtrain -= mean
    std = xtrain.std(axis=0)
    xtrain /= std
    xtest=X_test
    xtest -= mean
    xtest /= std
    model.fit(xtrain, ytrain,
              epochs=12, batch_size=96, verbose=0,callbacks=[tensorboard_callback])
    model.save_weights('pilot.weights.h5')
    test_mse_score, test_mae_score = model.evaluate(xtest,ytest)
    print(f'test_mse_score,test_mae_score -> {test_mse_score}, {test_mae_score}')
    model.load_weights('pilot.weights.h5')
    predictions = model.predict(xtest)
    correct=0
    for i in range(len(predictions)):
        print(f'prediction {predictions[i]}->actual {ytest[i]}')
        if predictions[i]<=ytest[i]:    
            correct+=1
    print(correct)
    print(correct/len(Y_test)*100)

In [45]:
train(shape=(1,))

Model: "functional_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_17 (InputLayer)          │ (None, 1)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_68 (Dense)                     │ (None, 32)                  │              64 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_69 (Dense)                     │ (None, 64)                  │           2,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_70 (Dense)                     │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_71 (Dense)                     │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,401 (25.00 KB)

 Trainable params: 6,401 (25.00 KB)

 Non-trainable params: 0 (0.00 B)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 34.2006 - mae: 1.4107 
test_mse_score,test_mae_score -> 123.4742202758789, 3.4538230895996094
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
prediction [1.5931073]->actual 1.0
prediction [1.5581306]->actual 1.0
prediction [1.3326315]->actual 1.0
prediction [1.2900283]->actual 1.0
prediction [1.5882596]->actual 1.01
prediction [1.3783165]->actual 1.01
prediction [1.444198]->actual 1.02
prediction [1.3049645]->actual 1.02
prediction [1.4389153]->actual 1.04
prediction [1.5519794]->actual 1.05
prediction [1.3122889]->actual 1.05
prediction [1.2731813]->actual 1.05
prediction [1.0988636]->actual 1.05
prediction [1.6158439]->actual 1.07
prediction [1.1683179]->actual 1.07
prediction [1.5571209]->actual 1.08
prediction [1.2364763]->actual 1.08
prediction [1.5061244]->actual 1.1
prediction [1.179145]->actual 1.1
prediction [1.1568161]->actual 1.1
prediction [1.5560031]->actual 1.12
prediction [1.3452623]->actual 1.12
prediction [1.3392118]->actual 1.12
pred

In [23]:
# Clear any logs from previous runs
#rm -rf ./logs/

In [24]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 15220), started 1 day, 0:37:49 ago. (Use '!kill 15220' to kill it.)